In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_date, date_format

spark = SparkSession.builder.appName("FirstLastTxn").getOrCreate()

# Sample Data
data = [
    ("C1", "2024-01-01", 100),
    ("C1", "2024-01-05", 200),
    ("C1", "2024-01-10", 300),
    ("C2", "2024-01-02", 400),
    ("C2", "2024-01-20", 500)
]

columns = ["customer_id", "txn_date", "amount"]

df = spark.createDataFrame(data, columns).withColumn("txn_date", to_date("txn_date"))

# Extract month
df = df.withColumn("month", date_format("txn_date", "yyyy-MM"))

# Window for first txn
window_first = Window.partitionBy("customer_id", "month").orderBy("txn_date")

# Window for last txn
window_last = Window.partitionBy("customer_id", "month").orderBy(col("txn_date").desc())

df = df.withColumn("rn_first", row_number().over(window_first))
df = df.withColumn("rn_last", row_number().over(window_last))

# Filter first and last
result_df = df.filter((col("rn_first") == 1) | (col("rn_last") == 1))

result_df.show()